In [0]:
import os
from pyspark.sql.functions import current_timestamp, input_file_name, lit

# 1. Identifica o catálogo ativo e localiza o volume
catalogo = spark.catalog.currentCatalog()
pasta_volume = f"/Volumes/{catalogo}/bronze/raw_files"

# 2. Lista os arquivos presentes no volume
arquivos = [f.name for f in dbutils.fs.ls(pasta_volume) if f.name.endswith(".csv")]
print(f"Arquivos CSV encontrados ({len(arquivos)}): {arquivos}")

# 3. Itera sobre cada CSV e salva como tabela Delta na Bronze
for nome_arquivo in arquivos:
  caminho_csv = f"{pasta_volume}/{nome_arquivo}"

  # Define o nome da tabela (remove a extensão .csv e padroniza para minúsculas)
  nome_tabela = os.path.splitext(nome_arquivo)[0].lower().replace("-", "_")

  print(f"Ingerindo '{nome_arquivo}' -> bronze.{nome_tabela}...")

  # Lê o CSV inferindo tipos e cabeçalho
  df_csv = (
      spark.read.option("header", "true")
      .option("inferSchema", "true")
      .csv(caminho_csv)
      .withColumn("_ingested_at", current_timestamp())
      .withColumn("_source_file", lit(nome_arquivo))
  )

  # Grava no formato Delta Lake gerenciado
  df_csv.write.format("delta").mode("overwrite").saveAsTable(
      f"bronze.{nome_tabela}"
  )

  qtd_linhas = spark.table(f"bronze.{nome_tabela}").count()
  print(f"[OK] Tabela 'bronze.{nome_tabela}' criada com {qtd_linhas} linhas.")

print("\nTodas as tabelas foram gravadas com sucesso no schema bronze.")

In [0]:
catalogo = spark.catalog.currentCatalog()
print(f"=== DIAGNÓSTICO DO AMBIENTE ===")
print(f"Catálogo ativo: {catalogo}\n")

# 1. Schemas existentes
schemas = [s.name for s in spark.catalog.listDatabases()]
print(f"Schemas encontrados: {schemas}")

# 2. Volumes no schema bronze
try:
  volumes = spark.sql(f"SHOW VOLUMES IN {catalogo}.bronze").collect()
  print(f"Volumes em 'bronze': {[v['volume_name'] for v in volumes]}")

  # Lista arquivos dentro do volume raw_files se existir
  arquivos_volume = dbutils.fs.ls(f"/Volumes/{catalogo}/bronze/raw_files")
  print(f"Arquivos no Volume raw_files:")
  for f in arquivos_volume:
    print(f"  - {f.name} ({round(f.size / (1024*1024), 2)} MB)")
except Exception as e:
  print(f"Aviso sobre Volumes: {e}")

# 3. Tabelas Delta registradas no schema bronze
try:
  tabelas_bronze = spark.sql(f"SHOW TABLES IN {catalogo}.bronze").collect()
  print(
      f"\nTabelas Delta em 'bronze' ({len(tabelas_bronze)} tabelas encontradas):"
  )
  for t in tabelas_bronze:
    print(f"  - {t['tableName']}")
except Exception as e:
  print(f"Aviso sobre Tabelas: {e}")

In [0]:
print('oi')